In [11]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib

# Force interactive popup window for dragging/rotating
matplotlib.use('TkAgg') 

def get_air_density(altitude):
    return 1.225 * np.exp(-max(altitude, 0.0) / 8500.0)

def simulate_aerospace_interception():
    dt = 0.1
    max_time = 90.0
    g = np.array([0.0, 0.0, -9.81])
    
    # --- MISSILE SETUP ---
    targets = [
        {'id': 0, 'name': 'Main Bus', 'pos': np.array([12000.0, 12000.0, 0.0]), 'vel': np.array([-380.0, -380.0, 750.0]), 
         'active': True, 'mass': 1500.0, 'area': 0.8, 'history': [], 'color': 'darkred'},
        {'id': 1, 'name': 'MIRV-1', 'pos': np.zeros(3, dtype=float), 'vel': np.zeros(3, dtype=float), 
         'active': False, 'mass': 200.0, 'area': 0.1, 'history': [], 'color': 'red'},
        {'id': 2, 'name': 'MIRV-2', 'pos': np.zeros(3, dtype=float), 'vel': np.zeros(3, dtype=float), 
         'active': False, 'mass': 200.0, 'area': 0.1, 'history': [], 'color': 'red'},
        {'id': 3, 'name': 'MIRV-3', 'pos': np.zeros(3, dtype=float), 'vel': np.zeros(3, dtype=float), 
         'active': False, 'mass': 200.0, 'area': 0.1, 'history': [], 'color': 'red'}
    ]
    mirv_deploy_time = 14.0 
    
    # --- INTERCEPTORS SETUP ---
    interceptors = [
        {'id': 1, 'name': 'INT-1', 'pos': np.array([0.0, 0.0, 0.0]), 'vel': np.array([0.0, 0.0, 0.0]),
         'delay': 5.0, 'target_idx': 0, 'color': 'blue', 'history': [], 'hit': False}, 
        {'id': 2, 'name': 'INT-2', 'pos': np.array([4000.0, -2000.0, 0.0]), 'vel': np.array([0.0, 0.0, 0.0]),
         'delay': 8.0, 'target_idx': 0, 'color': 'darkblue', 'history': [], 'hit': False},
        {'id': 3, 'name': 'INT-3', 'pos': np.array([-2000.0, 3000.0, 0.0]), 'vel': np.array([0.0, 0.0, 0.0]),
         'delay': 11.0, 'target_idx': 0, 'color': 'dodgerblue', 'history': [], 'hit': False}
    ]
    
    # Physics Constants
    initial_mass_i, dry_mass_i = 350.0, 150.0            
    motor_burn_time = 20.0        
    mass_burn_rate = (initial_mass_i - dry_mass_i) / motor_burn_time
    thrust_force = 75000.0 
    
    area_i, cd_i, cd_t = 0.15, 0.4, 0.3            
    N_nav = 4.5           
    hit_radius = 50.0        
    radar_noise_std = 50.0   
    max_i_g_force = 45.0 * 9.81 
    vls_time = 2.0        
    
    # History init
    for tgt in targets: tgt['history'].append(tgt['pos'].copy() if tgt['active'] else np.array([np.nan]*3))
    for icpt in interceptors: 
        icpt['history'].append(icpt['pos'].copy())
        icpt['x_est'], icpt['v_est'] = None, None

    t = 0.0
    while t < max_time:
        if t >= mirv_deploy_time and targets[0]['active']:
            targets[0]['active'] = False 
            bus_pos, bus_vel = targets[0]['pos'], targets[0]['vel']
            for i in range(1, 4):
                targets[i]['active'] = True
                targets[i]['pos'] = bus_pos.copy()
                spread = np.array([np.cos(i*2.1)*70, np.sin(i*2.1)*70, 30.0])
                targets[i]['vel'] = bus_vel.copy() + spread
            interceptors[0]['target_idx'], interceptors[1]['target_idx'], interceptors[2]['target_idx'] = 1, 2, 3
            for icpt in interceptors: icpt['x_est'], icpt['v_est'] = None, None

        for tgt in targets:
            if not tgt['active']: 
                tgt['history'].append(np.array([np.nan]*3))
                continue
            rho = get_air_density(tgt['pos'][2])
            speed = np.linalg.norm(tgt['vel'])
            drag = -(0.5 * rho * speed**2 * cd_t * tgt['area'] / tgt['mass']) * (tgt['vel']/speed) if speed > 1e-3 else np.zeros(3)
            evasion = np.zeros(3)
            if tgt['id'] > 0 and tgt['pos'][2] < 6000.0:
                ortho = np.cross(tgt['vel'], np.array([np.cos(t*2), np.sin(t*2), 0.0]))
                if np.linalg.norm(ortho) > 1e-3: evasion = (ortho / np.linalg.norm(ortho)) * (15.0 * 9.81)
            tgt['vel'] += (g + drag + evasion) * dt
            tgt['pos'] += tgt['vel'] * dt
            tgt['history'].append(tgt['pos'].copy())

        for icpt in interceptors:
            if t >= icpt['delay'] and not icpt['hit']:
                time_active = t - icpt['delay']
                if np.linalg.norm(icpt['vel']) < 1e-3: icpt['vel'] = np.array([0.0, 0.0, 10.0])
                
                tgt = targets[icpt['target_idx']]
                if not tgt['active']:
                    accel_i = g
                else:
                    meas = tgt['pos'] + np.random.normal(0, radar_noise_std, 3)
                    if icpt['x_est'] is None:
                        icpt['x_est'], icpt['v_est'] = meas, tgt['vel'].astype(float)
                    else:
                        pred_x = icpt['x_est'] + icpt['v_est'] * dt
                        res = meas - pred_x
                        icpt['x_est'], icpt['v_est'] = pred_x + 0.6*res, icpt['v_est'] + (0.1/dt)*res

                    curr_mass = max(dry_mass_i, initial_mass_i - mass_burn_rate * time_active)
                    v_mag = np.linalg.norm(icpt['vel'])
                    heading = icpt['vel'] / v_mag if v_mag > 1e-3 else np.array([0.0, 0.0, 1.0])
                    thrust = heading * (thrust_force/curr_mass) if time_active < motor_burn_time else np.zeros(3)
                    rho = get_air_density(icpt['pos'][2])
                    drag = -(0.5 * rho * v_mag**2 * cd_i * area_i / curr_mass) * heading
                    
                    R, V = icpt['x_est'] - icpt['pos'], icpt['v_est'] - icpt['vel']
                    if np.dot(heading, R) < 0: lat_accel = np.zeros(3)
                    else:
                        Omega = np.cross(R, V) / np.dot(R, R)
                        lat_accel = N_nav * np.linalg.norm(V) * np.cross(Omega, heading)
                        if np.linalg.norm(lat_accel) > max_i_g_force: lat_accel = (lat_accel / np.linalg.norm(lat_accel)) * max_i_g_force
                    accel_i = (np.array([0.0, 0.0, 250.0]) if time_active < vls_time else (lat_accel + thrust)) + g + drag
                
                if tgt['active'] and np.linalg.norm(tgt['pos'] - icpt['pos']) < hit_radius:
                    icpt['hit'], tgt['active'] = True, False

                icpt['vel'] = icpt['vel'].astype(float) + (accel_i * dt)
                icpt['pos'] = icpt['pos'].astype(float) + (icpt['vel'] * dt)
                if icpt['pos'][2] < 0: icpt['pos'][2] = 0.0
            icpt['history'].append(icpt['pos'].copy())
        t += dt
    return targets, interceptors, dt

# --- Run & Visualize ---
targets, interceptors, dt = simulate_aerospace_interception()
frames = len(interceptors[0]['history'])

fig = plt.figure(figsize=(14, 9))
ax = fig.add_subplot(111, projection='3d')

# HUD: Only Time
status_text = ax.text2D(0.02, 0.95, "", transform=ax.transAxes, fontsize=14, 
                        bbox=dict(facecolor='black', alpha=0.8, edgecolor='white'))

# Setup Lines
lines_t = [ax.plot([], [], [], color=t['color'], lw=2, label=t['name'])[0] for t in targets]
pts_t = [ax.plot([], [], [], marker='o', color=t['color'])[0] for t in targets]

for i in interceptors:
    i['l'], = ax.plot([], [], [], color=i['color'], ls='--', label=i['name'])
    i['p'], = ax.plot([], [], [], marker='^', color=i['color'])

ax.set_xlim(-5000, 15000); ax.set_ylim(-5000, 15000); ax.set_zlim(0, 8000)
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_zlabel('Alt (m)')
ax.legend(loc='upper right', fontsize=10)

def update(frame):
    current_time = frame * dt
    for idx, t in enumerate(targets):
        h = np.array(t['history'])
        lines_t[idx].set_data(h[:frame, 0], h[:frame, 1])
        lines_t[idx].set_3d_properties(h[:frame, 2])
        pts_t[idx].set_data([h[frame-1, 0]], [h[frame-1, 1]])
        pts_t[idx].set_3d_properties([h[frame-1, 2]])
    
    for i in interceptors:
        h = np.array(i['history'])
        i['l'].set_data(h[:frame, 0], h[:frame, 1])
        i['l'].set_3d_properties(h[:frame, 2])
        i['p'].set_data([h[frame-1, 0]], [h[frame-1, 1]])
        i['p'].set_3d_properties([h[frame-1, 2]])
    
    status_text.set_text(f"T + {current_time:.1f} s")
    status_text.set_color('lime')
    ax.view_init(elev=15, azim=ax.azim)
    return status_text,

ani = animation.FuncAnimation(fig, update, frames=range(1, frames, 3), interval=10, blit=False)
plt.show()